In [ ]:

storage_account_name = "REPLACE_WITH_ACTUAL_KEY"
storage_account_key = "REPLACE_WITH_ACTUAL_KEY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", 
    storage_account_key
)

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType
import logging


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [%(levelname)s] - %(name)s: %(message)s',
    force=True )

In [ ]:
def string_cleaning(df):

    cleaned_df = df.select(*[
        trim(c).alias(c) if df.schema[c].dataType.typeName()=='string'
        else col(c)
        for c in df.columns
    ])
    return cleaned_df

In [0]:
bronze_path = "abfss://olistdb@iliststorageaccount.dfs.core.windows.net/bronze/"
silver_path = "abfss://olistdb@iliststorageaccount.dfs.core.windows.net/silver/"

schema =  {

                'bronze_customers' : StructType([
                    StructField('customer_id', StringType(), False),
                    StructField('customer_unique_id', StringType(), False),
                    StructField('customer_zip_code_prefix', IntegerType(), True),
                    StructField('customer_city', StringType(), True),
                    StructField('customer_state', StringType(), True, )]),
                'bronze_geolocation': StructType([
                    StructField('geolocation_zip_code_prefix', IntegerType(), True),
                    StructField('geolocation_lat', DoubleType(), True),
                    StructField('geolocation_lng', DoubleType(), True),
                    StructField('geolocation_city', StringType(), True),
                    StructField('geolocation_state', StringType(), True)
                    ]),
                
                'bronze_order_items': StructType([
                    StructField('order_id', StringType(), False),
                    StructField('order_item_id', IntegerType(), False),
                    StructField('product_id', StringType(), False),
                    StructField('seller_id', StringType(), False),
                    StructField('shipping_limit_date', TimestampType(), True),
                    StructField('price', DoubleType(), True),
                    StructField('freight_value', DoubleType(), True)
                    ]),
                'bronze_order_payments': StructType([
                    StructField('order_id', StringType(), False),
                    StructField('payment_sequential', IntegerType(), False),
                    StructField('payment_type', StringType(), False),
                    StructField('payment_installments', IntegerType(), True),
                    StructField('payment_value', DoubleType(), True)
                    ]),
                'bronze_order_reviews': StructType([
                    StructField('review_id', StringType(), False),
                    StructField('order_id', StringType(), False),
                    StructField('review_score', IntegerType(), True),
                    StructField('review_comment_title', StringType(), True),
                    StructField('review_comment_message', StringType(), True),
                    StructField('review_creation_date', TimestampType(), True),
                    StructField('review_answer_timestamp', TimestampType(), True)
                    ]),
                'bronze_orders': StructType([
                    StructField('order_id', StringType(), False),
                    StructField('customer_id', StringType(), False),
                    StructField('order_status', StringType(), False),
                    StructField('order_purchase_timestamp', TimestampType(), True),
                    StructField('order_approved_at', TimestampType(), True),
                    StructField('order_delivered_carrier_date', TimestampType(), True),
                    StructField('order_delivered_customer_date', TimestampType(), True),
                    StructField('order_estimated_delivery_date', TimestampType(), True)
                    ]),
                'bronze_products': StructType([
                    StructField('product_id', StringType(), False),
                    StructField('product_category_name', StringType(), False),
                    StructField('product_name_lenght', IntegerType(), True),
                    StructField('product_description_lenght', IntegerType(), True),
                    StructField('product_photos_qty', IntegerType(), True),
                    StructField('product_weight_g', IntegerType(), True),
                    StructField('product_length_cm', IntegerType(), True),
                    StructField('product_height_cm', IntegerType(), True),
                    StructField('product_width_cm', IntegerType(), True)
                    ]),
                'bronze_sellers': StructType([
                    StructField('seller_id', StringType(), False),
                    StructField('seller_zip_code_prefix', IntegerType(), False),
                    StructField('seller_city', StringType(), False),
                    StructField('seller_state', StringType(), False)
                    ]),
                'bronze_product_category_name_translation': StructType([
                    StructField('product_category_name', StringType(), False),
                    StructField('product_category_name_english', StringType(), False)
                    ]),

}

tables_ds = [{"bronze_customers":bronze_path+"olist_customers_dataset.csv",
             "bronze_geolocation":bronze_path+"olist_geolocation_dataset.csv",
             "bronze_order_items":bronze_path+"olist_order_items_dataset.csv",
             "bronze_order_payments":bronze_path+"olist_order_payments",
             "bronze_order_reviews":bronze_path+"olist_order_reviews_dataset.csv",
             "bronze_orders":bronze_path+"olist_orders_dataset.csv",
             "bronze_products":bronze_path+"olist_products_dataset.csv",
             "bronze_sellers":bronze_path+"olist_sellers_dataset.csv",
             "bronze_product_category_name_translation":bronze_path+"product_category_name_translation.csv"}
             ]

def write_df(table_path, table_name):
    df_name=(spark.read
    .format('csv')
    .option("header", "true")
    .schema(schema[table_name])
    .load(table_path))
    return df_name

for table in tables_ds:
    for table_name , path in table.items():
        globals()[table_name] = write_df(path,table_name)


In [0]:
def null_check(df_name):
    print(df_name)
    new_df = df_name.select([count(when(col(c).isNull(),1)).alias(c) for c in df_name.columns])
    new_df.show(n=1 , vertical=True)

for table in tables_ds:
    for table_name , path in table.items():
        df_name = globals().get(table_name)
        null_check(df_name)


In [0]:
## take average lat and lng for each zip code
geolocation_logger = logging.getLogger("bronze_geolocation")

try:
    
    geolocation_logger.info("start transformation")
    silver_geolocation = (bronze_geolocation
                        .filter(col('geolocation_zip_code_prefix').isNotNull())
                        .groupBy('geolocation_zip_code_prefix').agg(avg('geolocation_lat').alias('geolocation_lat'),
                                avg('geolocation_lng').alias('geolocation_lng'))        
                            )

    geolocation_logger.info("Transformation done")
except Exception as e:
    geolocation_logger.exception(f"Transformation failed:{e}" , exc_info=True)



In [0]:
cust_logger = logging.getLogger("bronze_customers")

try:
        cust_logger.info('start transformation')    
        silver_customers =  (bronze_customers
                        
                        ## filtering (null handling)
                        .filter(col('customer_unique_id').isNotNull())
                        
                        ## deduplicatinon
                        .dropDuplicates(["customer_id"])
        

                        ## standerization (trim)
                        .transform(string_cleaning)
                        
                        ## metadata  (join ,  add columns)
                        .join(silver_geolocation.select('geolocation_lat' ,'geolocation_zip_code_prefix', "geolocation_lng") , bronze_customers.customer_zip_code_prefix==silver_geolocation.geolocation_zip_code_prefix , how='left' )
                        .drop('geolocation_zip_code_prefix')
                        .withColumn('created_at', current_timestamp())
                        )
        cust_logger.info('transformation done')
except Exception as e:
        cust_logger.exception(f'transformation failed:{e}' , exc_info=True)

In [0]:
seller_logger = logging.getLogger("bronze_sellers")
try:
    seller_logger.info('start transformation')
    silver_sellers  =  (
                            bronze_sellers
                            .filter(col('seller_id').isNotNull())

                            .dropDuplicates(["seller_id"])

                            .transform(string_cleaning)
                                    

                            .join(silver_geolocation.select('geolocation_lat' ,'geolocation_zip_code_prefix', "geolocation_lng") , bronze_sellers.seller_zip_code_prefix==silver_geolocation.geolocation_zip_code_prefix , how='left').drop('geolocation_zip_code_prefix')
                            .withColumn('created_at', current_timestamp())              
                        )
    seller_logger.info('transformation done')
except Exception as e:
    seller_logger.exception(f'transformation failed:{e}' , exc_info=True)




In [0]:
product_category_name_translation_logger = logging.getLogger("bronze_product_category_name_translation")
try:
    product_category_name_translation_logger.info('start transformation') 
    silver_product_category_name_translation = (

                                                bronze_product_category_name_translation
                                                .filter(col('product_category_name').isNotNull())
                                                .filter(col('product_category_name_english').isNotNull())

                                                .dropDuplicates(["product_category_name"])
                                                .transform(string_cleaning)

    )
    product_category_name_translation_logger.info('transformation done')
except Exception as e:
    product_category_name_translation_logger.exception(f'transformation failed:{e}' , exc_info=True)



In [0]:
product_logger=logging.getLogger("bronze_products")
try:
    product_logger.info('start transformation')
    silver_products = ( bronze_products
                    .filter(col('product_id').isNotNull())
                    .filter(col('product_category_name').isNotNull())

                    .dropDuplicates(["product_id"])
                    .transform(string_cleaning)

                    .join(silver_product_category_name_translation , on="product_category_name",how="left").drop("product_category_name")

                    .withColumn('created_at', current_timestamp())      
                    )
    product_logger.info('transformation done')
except Exception as e:
    product_logger.exception(f'transformation failed:{e}' , exc_info=True)




In [0]:
order_logger = logging.getLogger("order_logger")

try:
    order_logger.info("start transformation")
    silver_orders = (bronze_orders
                    .filter(col('order_id').isNotNull())
                    
                    .dropDuplicates()

                    .transform(string_cleaning)

                    .withColumn("approved_performance_hour", f.round(((f.col("order_approved_at").cast("long") - f.col("order_purchase_timestamp").cast("long")) / 3600), 2))
                    .withColumn( "total_process_days",
                        when(
                            col("order_delivered_customer_date") > col("order_purchase_timestamp"),
                            datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
                            ).otherwise(lit(None))
                            )
                    .withColumn('delivery_performance_days', datediff(col('order_estimated_delivery_date'), col('order_delivered_customer_date')))

                    .withColumn('order_purchase_date', col('order_purchase_timestamp').cast('Date'))
                    .withColumn('order_approved_date', col('order_approved_at').cast('Date'))
                    .withColumn('order_delivered_carrier_date_d', col('order_delivered_carrier_date').cast('Date'))
                    .withColumn('order_delivered_customer_date_d', col('order_delivered_customer_date').cast('Date'))

                    .withColumn('order_purchase_time', date_format(col('order_purchase_timestamp'), "HH:mm:ss"))
                    .withColumn('order_approved_time', date_format(col('order_approved_at'), "HH:mm:ss"))
                    .withColumn('order_delivered_carrier_time', date_format(col('order_delivered_carrier_date'), "HH:mm:ss"))
                    .withColumn('order_delivered_customer_time', date_format(col('order_delivered_customer_date'), "HH:mm:ss"))

                    .withColumn('red_flag', 
                                when(
                                    (
                                        (col('order_status') == 'delivered') & 
                                        (
                                            col('order_delivered_customer_date').isNull() |
                                            col("order_approved_at").isNull() |
                                            col('order_delivered_carrier_date').isNull() |
                                            col('order_purchase_timestamp').isNull() |
                                            col('order_estimated_delivery_date').isNull()
                                        )
                                    ) | 
                                    (
                                        (col('order_status') == 'canceled') & 
                                        (col('order_delivered_customer_date').isNotNull())
                                    ) | 
                                    (col('order_delivered_customer_date') < col('order_purchase_timestamp')), 
                                    lit(1)
                                ).otherwise(lit(0))
                    )

                    .withColumn('created_at', current_timestamp())              
                    )
       
    order_logger.info("transformation done")

except Exception as e:
    order_logger.error(f"transformation failed: {e}", exc_info=True)

    

In [0]:
quar_logger = logging.getLogger("quarantine_logger")
try :
    quar_logger.info("pulling start")

    silver_quarantine = bronze_orders.filter(
        col('order_id').isNull() | 
        col('customer_id').isNull() | 
        col('order_status').isNull()
    )
    quar_logger.info("pulling finish")
except Exception as e:
    quar_logger.error(f"pulling failed: {e}", exc_info=True)

In [0]:
payments_logger= logging.getLogger("payments_logger")
try:
    payments_logger.info("pulling start")    
    silver_order_payments = (bronze_order_payments
                            .filter(col('order-_id').isNotNull())
                            .transform(string_cleaning)
                            .withColumn('created_at', current_timestamp())
                            .withColumn('payment_sequential' , 
                                        when(
                                            col('payment_sequential')==0
                                            , lit(1)
                                            ).otherwise(col('payment_sequential'))
                                        )
                            .withColumn('payment_installments' , 
                                        when(
                                            col('payment_installments')==0
                                            , lit(1)
                                            ).otherwise(col('payment_installments'))
                                        )
                            )
    payments_logger.info("pulling finish")
except Exception as e:
    payments_logger.error(f"pulling failed: {e}" , exc_info=True)

In [0]:
review_logger= logging.getLogger("review_logger")
try:
    review_logger.info("pulling start")
    window_spec_review = Window.partitionBy('order_id').orderBy(col('review_answer_timestamp').desc())

    silver_order_reviews = (bronze_order_reviews
                            .withColumn('review_score',expr("try_cast(review_score as int)"))

                            .withColumn('review_creation_date',expr("try_cast(review_creation_date as date)"))

                            
                            .filter(col('order_id').isNotNull())
                            .filter(col('review_score').between(1,5))
                            .transform(string_cleaning)

                            .withColumn('rank',row_number().over(window_spec_review))
                            .filter(col('rank')==1).drop('rank')

                            .withColumn('created_at', current_timestamp())                        
                            )
    review_logger.info("pulling finish")
except Exception as e:
    review_logger.error(f"pulling failed: {e}" , exc_info=True)



In [0]:

order_item_logger= logging.getLogger("order_item_logger")

try:
    order_item_logger.info("pulling start")
    silver_order_items = (
        bronze_order_items

        .dropDuplicates()

        .transform(string_cleaning)



        .withColumn('shipping_limit_date', expr("try_cast(shipping_limit_date as timestamp)"))
        .withColumn('freight_value',expr("try_cast(freight_value as DECIMAL(10,2))"))


        .groupby('order_id','product_id')
        .agg(
                f.count('*').alias('Total_QTY'),
                f.first('price').alias('Unit_price'),
                f.first('freight_value').alias('Unit_freight'),
                f.first('seller_id').alias('seller_id'),
                f.first('shipping_limit_date').alias('shipping_limit_date'), 
                f.sum('price').alias('Total_product_price'),
                f.sum('freight_value').alias('Total_freight')
        )
        .withColumn('Total_order_value',col('Total_product_price')+col('Total_freight'))

        .withColumn('created_at', current_timestamp())
    )
    order_item_logger.info("pulling finish")
except Exception as e:
    order_item_logger.error(f"pulling failed: {e}" , exc_info=True)




In [0]:

df_list_for_silver = [
    (silver_order_reviews, "silver_order_reviews"),
    (silver_order_payments, "silver_order_payments"),
    (silver_order_items, "silver_order_items"),
    (silver_quarantine, "silver_quarantine_orders"),
    (silver_orders, "silver_orders"),
    (silver_products, "silver_products"),
    (silver_product_category_name_translation, "silver_product_category_translation"),
    (silver_sellers, "silver_sellers"),
    (silver_customers, "silver_customers"),
    (silver_geolocation, "silver_geolocation")
]

zorder_list_dict = {
    "silver_orders": "order_id, customer_id",
    "silver_order_items": "order_id, product_id",
    "silver_products": "product_id",
    "silver_customers": "customer_id",
    "silver_sellers": "seller_id"
}

save_to_silver_logger = logging.getLogger("save_to_silver_logger")


def save_to_silver (df_list ,targer_path, zorder_list=None):
    try:
        save_to_silver_logger.info("start saving to silver")
        for df, table_name in df_list:

            save_to_silver_logger.info(f"start saving to silver : {table_name}")
            spark.sql(f"DROP TABLE IF EXISTS default.{table_name}")
            (df.write.format("delta")
                    .option("path",targer_path+table_name)
                    .mode("overwrite")
                    .option("overwriteSchema","true")
                    .saveAsTable(table_name))
            
            if table_name in zorder_list:
                keys = zorder_list[table_name]
                save_to_silver_logger.info(f"start zorder for table {table_name} with keys {keys}")
                spark.sql(f"OPTIMIZE default.{table_name} ZORDER BY ({keys})")
                save_to_silver_logger.info(f"finish saving to silver and zordering table:{table_name}")
            else :
                save_to_silver_logger.info(f"finish saving to silver table : {table_name}")


    except Exception as e:
        save_to_silver_logger.error(f"failed to save to silver: {e}" , exc_info=True)


save_to_silver(df_list_for_silver,silver_path,zorder_list_dict)



